# Case 8 — RAG Pipeline

Case 7, kural motorunun kararlarını yapılandırılmış (rule_id, severity, koşul, şablon mesaj)
biçimde açıkladı. Case 8, bunu **bilgi destekli, doğal dilde** açıklanabilirliğe taşıyor: örnek
fraud-policy dökümanlarından bir knowledge base kuruyoruz, bir soru/işlem için ilgili policy
metnini geri getiriyoruz (retrieval), bunu bir LLM'e bağlam olarak enjekte ediyoruz (context
injection), ve politika-temelli bir açıklama üretiyoruz (RAG reasoning).

**Kritik kısıtlama:** brief, Case 8/9 için sadece **local LLM** (Ollama) kullanılmasını istiyor —
ücretli API anahtarı gerektirmemesi için. Ollama bu makinede henüz kurulu değil (sudo gerektiriyor).
Bu yüzden mimari, embedding için iki gerçek strateji destekliyor: `TfidfEmbeddingProvider`
(scikit-learn tabanlı, tamamen yerel, Ollama gerektirmez — test mock'u DEĞİL, kendi başına geçerli
bir yöntem) ve `OllamaEmbeddingProvider` (Ollama kurulunca kullanılacak). Bu notebook, Ollama
olmadan doğrulanabilecek HER ŞEYİ (chunking, DB, vector search, retrieval sıralaması, prompt
inşası) TF-IDF ile gerçekten çalıştırıp doğruluyor; sadece gerçek LLM üretimi Ollama kurulana kadar
beklemede kalıyor (zarif düşüş ile — çökmüyor, retrieval+prompt'u döndürüyor).

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "requirements.txt").exists():
            return parent
    raise RuntimeError("repo root not found — expected a requirements.txt somewhere above " + str(start))


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

PosixPath('/home/canberk/workspace/case_study')

In [2]:
import pandas as pd

from src.database.db import run_migrations
from src.database.db_services import new_session, close
from src.database.db_services.rag import clear_knowledge_base, list_chunks, list_documents

run_migrations()
print("rag_documents / rag_document_chunks tabloları hazır")

rag_documents / rag_document_chunks tabloları hazır


## 1. Knowledge Base — dökümanları yükleme

`data/knowledge_base/*.md`: 6 gerçek policy dökümanı (Case 6/7'de zaten kurulan, veriyle
doğrulanmış bulguların düzyazıya çevrilmiş hâli) + 2 **deneysel** döküman (brief'in izin verdiği
"var olmayan kurallar" — rule engine'de karşılığı yok, sadece RAG'ın kod tabanından bağımsız
çalıştığını göstermek için, açıkça `[DENEYSEL]` etiketli).

In [3]:
kb_dir = REPO_ROOT / "data" / "knowledge_base"

documents = []
for path in sorted(kb_dir.glob("*.md")):
    text = path.read_text(encoding="utf-8")
    title = text.splitlines()[0].lstrip("# ").strip()
    source = "experimental" if "experimental" in path.stem else "case_06_07"
    documents.append((title, source, text))

for title, source, text in documents:
    print(f"[{source:>12}] {title}")
print(f"\ntoplam: {len(documents)} döküman")

[  case_06_07] Mesai Saatleri Risk Politikası
[  case_06_07] Cihaz Parmak İzi Politikası
[experimental] [DENEYSEL — Rule Engine'de Karşılığı Yok] Kripto Borsası İşlem Politikası
[experimental] [DENEYSEL — Rule Engine'de Karşılığı Yok] Hediye Kartı Toplu Alım Politikası
[  case_06_07] Coğrafi Risk Politikası
[  case_06_07] Yeni Kart, Yüksek Değerli İlk İşlem Politikası
[  case_06_07] Güvenilir Kart (Trusted Entity) Politikası — Beklenmedik Bir Bulgu
[  case_06_07] Velocity (Hızlı Tekrar İşlem) Politikası

toplam: 8 döküman


## 2. Container — sağlayıcı seçimi

`RAGContainer`, `dependency_injector` ile `EMBEDDING_PROVIDER=tfidf|ollama` config'ine göre
`TfidfEmbeddingProvider`/`OllamaEmbeddingProvider` arasında seçim yapıyor (Strategy pattern +
Selector). Ollama kurulu olmadığı için `tfidf` ile başlıyoruz.

In [4]:
from src.services.rag.container import RAGContainer, DEFAULT_CONFIG
from src.services.rag.embeddings import ollama_is_running

container = RAGContainer()
container.config.from_dict(DEFAULT_CONFIG)
pipeline = container.rag_pipeline()

print("embedding provider:", pipeline.embedding_provider.name)
print("llm provider:", pipeline.llm_provider.name)
print("Ollama çalışıyor mu:", ollama_is_running())

embedding provider: tfidf
llm provider: ollama:smollm2:360m
Ollama çalışıyor mu: False


## 3. Embedding üretimi + storage — chunking, TF-IDF embedding, SQLite'a yazma

`ingest()`: her dökümanı chunk'lara böler (`chunking.py` — paragraf tabanlı, hedef ~120 kelime),
TÜM chunk'ları TEK SEFERDE embed eder (TF-IDF, tüm knowledge base'in kelime dağarcığına göre fit
edilmeli — Ollama'nın aksine, sabit önceden-eğitilmiş bir model değil), sonra her chunk'ı
embedding'iyle birlikte `rag_document_chunks` tablosuna yazar (`numpy.tobytes()` ile BLOB olarak).

`clear_knowledge_base()` önce çağrılır (idempotent re-run) — bunu doğrularken gerçek bir hata
buldum ve düzelttim: toplu `Query.delete()` SQLAlchemy'nin ORM cascade'ini tetiklemiyor, chunk'lar
silinmeden kalıp yeni ingest'te ID çakışmasıyla yanlış dökümanlara "yapışıyordu" — `db_services/
rag.py` artık her iki tabloyu da FK-sıralı şekilde açıkça siliyor.

In [5]:
db = new_session()

n_chunks = pipeline.ingest(db, documents)
print(f"{n_chunks} chunk oluşturuldu, embed edildi, DB'ye yazıldı")

# idempotency doğrulaması — aynı ingest'i tekrar çalıştır, yinelenme olmamalı
n_chunks_rerun = pipeline.ingest(db, documents)
all_chunks = list_chunks(db)
print(f"yeniden çalıştırma: {n_chunks_rerun} chunk üretildi, DB'de toplam {len(all_chunks)} chunk (yinelenme yok, doğru)")

9 chunk oluşturuldu, embed edildi, DB'ye yazıldı


yeniden çalıştırma: 9 chunk üretildi, DB'de toplam 9 chunk (yinelenme yok, doğru)


In [6]:
chunks_df = pd.DataFrame([
    {"chunk_id": c.id, "document": c.document.title, "chunk_index": c.chunk_index, "embedding_dim": c.embedding_dim, "text_preview": c.text[:60].replace("\n", " ")}
    for c in list_chunks(db)
])
chunks_df

,chunk_id,document,chunk_index,embedding_dim,text_preview
0,1,Mesai Saatleri Risk Politikası,0,481,"# Mesai Saatleri Risk Politikası Sistem, işle..."
1,2,Cihaz Parmak İzi Politikası,0,481,# Cihaz Parmak İzi Politikası Cihaz bilgisi (...
2,3,[DENEYSEL — Rule Engine'de Karşılığı Yok] Krip...,0,481,# [DENEYSEL — Rule Engine'de Karşılığı Yok] Kr...
3,4,[DENEYSEL — Rule Engine'de Karşılığı Yok] Hedi...,0,481,# [DENEYSEL — Rule Engine'de Karşılığı Yok] He...
4,5,Coğrafi Risk Politikası,0,481,# Coğrafi Risk Politikası Faturalandırma bölg...
5,6,Coğrafi Risk Politikası,1,481,"Diğer coğrafi adaylar (fiziksel mesafe, e-post..."
6,7,"Yeni Kart, Yüksek Değerli İlk İşlem Politikası",0,481,"# Yeni Kart, Yüksek Değerli İlk İşlem Politika..."
7,8,Güvenilir Kart (Trusted Entity) Politikası — B...,0,481,# Güvenilir Kart (Trusted Entity) Politikası —...
8,9,Velocity (Hızlı Tekrar İşlem) Politikası,0,481,# Velocity (Hızlı Tekrar İşlem) Politikası Ay...


## 4. Vector search — bilinen sorgularla doğrulama

Brute-force cosine similarity (`vector_search.py`, numpy) — bu ölçekte (birkaç düzine chunk) ANN
index'e gerek yok. Dört farklı Türkçe sorgu, her biri açıkça bir policy dökümanına işaret ediyor —
doğru döküman en üstte çıkıyor mu diye elle doğruluyoruz.

In [7]:
test_queries = [
    "Yabancı ülkeden yapılan yüksek tutarlı gece işlemi neden riskli?",
    "Aynı kartla çok kısa sürede tekrar işlem yapmak neden şüpheli?",
    "Uzun işlem geçmişi olan bir kart güvenilir sayılır mı?",
    "Kripto borsasına yapılan transferler için özel bir kural var mı?",
]

for q in test_queries:
    print("SORGU:", q)
    for r in pipeline.retrieve(db, q, top_k=2):
        print(f"  [{r.score:.3f}] {r.chunk.document_title} ({r.chunk.document_source})")
    print()

SORGU: Yabancı ülkeden yapılan yüksek tutarlı gece işlemi neden riskli?
  [0.266] Coğrafi Risk Politikası (case_06_07)
  [0.175] Yeni Kart, Yüksek Değerli İlk İşlem Politikası (case_06_07)

SORGU: Aynı kartla çok kısa sürede tekrar işlem yapmak neden şüpheli?
  [0.431] Velocity (Hızlı Tekrar İşlem) Politikası (case_06_07)
  [0.039] Mesai Saatleri Risk Politikası (case_06_07)

SORGU: Uzun işlem geçmişi olan bir kart güvenilir sayılır mı?
  [0.276] Güvenilir Kart (Trusted Entity) Politikası — Beklenmedik Bir Bulgu (case_06_07)
  [0.157] Yeni Kart, Yüksek Değerli İlk İşlem Politikası (case_06_07)

SORGU: Kripto borsasına yapılan transferler için özel bir kural var mı?
  [0.300] [DENEYSEL — Rule Engine'de Karşılığı Yok] Kripto Borsası İşlem Politikası (experimental)
  [0.167] [DENEYSEL — Rule Engine'de Karşılığı Yok] Hediye Kartı Toplu Alım Politikası (experimental)



**Dördü de doğru dökümanı en üstte buluyor** — coğrafi risk, velocity, trusted entity, ve
deneysel kripto politikası sorguları sırasıyla kendi dökümanlarını en yüksek skorla getiriyor. TF-
IDF, kelime örtüşmesine dayandığı için mükemmel bir semantik anlayış sağlamıyor (örn. eş anlamlı
ama farklı kelimeler kullanan bir sorguyu kaçırabilir), ama bu 8 dökümanlık, konuları net şekilde
ayrışan knowledge base için gayet işlevsel.

## 5. LLM Context Injection — prompt inşası

In [8]:
from src.services.rag.prompt import PromptBuilder

example_query = "Yabancı ülkeden gece yapılan yüksek tutarlı işlem neden riskli sayılır?"
retrieved = pipeline.retrieve(db, example_query, top_k=3)
prompt = PromptBuilder().with_context(retrieved).with_question(example_query).build()
print(prompt)

You are a policy assistant for a fraud/anomaly detection system. Answer using ONLY the source texts given below. Do not invent anything not present in the sources; if the sources don't cover the question, say so explicitly. Cite which source(s) you relied on using numbers like [1], [2].

Sources:

[1] Coğrafi Risk Politikası (similarity score: 0.275)
# Coğrafi Risk Politikası

Faturalandırma bölge/ülke kodu (addr2), sistemde ölçülen en güçlü risk sinyalidir. Verinin
%99,2'si tek bir yerli bölge koduna aittir; bu koddan farklı (yabancı) işlemlerde fraud oranı
yerli işlemlere göre yaklaşık 4,26 kat daha yüksektir. Bölge kodu eksik olan işlemlerde ise fraud
oranı yabancı işlemlerden bile daha yüksektir (yaklaşık 4,91 kat) — bu yüzden politika, yabancı ve
eksik durumları AYRI iki risk katmanı olarak ele alır, tek bir "yabancı" kategorisinde birleştirmez.

İki yöntem uygulanır: sabit bir politika çarpanı (sadece kesin bilinen yabancı işlemler için, 2,0
kat — eksik veriye dokunulmaz, çünkü e

`PromptBuilder` (Builder pattern), sistem talimatı + retrieved chunk'ların GERÇEK metni + soruyu
tek bir prompt'ta birleştiriyor, her kaynağı numaralandırıp benzerlik skorunu da gösteriyor — bu,
LLM context injection'ın somut karşılığı: model, kendi "bilgisine" değil, buraya enjekte edilen
gerçek policy metnine dayanarak cevap vermeye yönlendiriliyor.

## 6. RAG Reasoning Akışı — uçtan uca, Ollama'nın zarif düşüşüyle

In [9]:
result = pipeline.answer(db, example_query)
print("SORU:", result["question"])
print("CEVAP:", result["answer"])
print("NOT:", result["note"])

SORU: Yabancı ülkeden gece yapılan yüksek tutarlı işlem neden riskli sayılır?
CEVAP: None
NOT: LLM generation failed (ollama:smollm2:360m): [Errno 111] Connection refused — showing retrieval + prompt only.


**Ollama kurulu olmadığı için `answer=None`, ama sistem ÇÖKMÜYOR** — `note` alanı neyin eksik
olduğunu açıkça söylüyor, `retrieve()` ve `prompt` inşası tam olarak çalışmaya devam ediyor.
`RAGPipeline.answer()`, LLM çağrısını `try/except httpx.HTTPError` ile sarıyor — sağlayıcıdan
bağımsız bir zarif düşüş, sadece Ollama'ya özel bir kontrol değil.

## 7. Case 7 Köprüsü — anomali sonuçlarını RAG ile açıklama

In [10]:
from src.services.rules.loader import RuleLoader
from src.services.rules.resolution import build_default_resolution_chain
from src.services.rules.engine import RuleEngine
import pyarrow.parquet as pq
from src.config import settings
from src.services.features.temporal import build_temporal_features
from src.services.features.entity import build_entity_features
from src.services.features.relational import build_relational_features
from src.services.anomaly.combined import compute_all_anomaly_scores, PRIMARY_SCORE_COLUMNS
from src.services.anomaly.normalization import normalize_scores
from src.services.anomaly.aggregation import compute_final_raw_anomaly_score

parquet_path = settings.processed_data_path / "merged_transactions.parquet"
raw = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "TransactionAmt", "addr2", "DeviceInfo", "dist1"]).to_pandas()
temporal = build_temporal_features(parquet_path)
entity = build_entity_features(parquet_path)
relational = build_relational_features(parquet_path)
all_scores = compute_all_anomaly_scores(parquet_path)
normalized = normalize_scores(all_scores, PRIMARY_SCORE_COLUMNS)
final_raw = compute_final_raw_anomaly_score(normalized, PRIMARY_SCORE_COLUMNS)

rules_df = raw.merge(temporal, on="TransactionID").merge(entity, on="TransactionID").merge(relational, on="TransactionID").merge(final_raw, on="TransactionID")

rule_loader = RuleLoader()
rules = rule_loader.load(REPO_ROOT / "src/services/rules/definitions/fraud_rules.yaml")
rule_engine = RuleEngine(rules, build_default_resolution_chain())
rule_result = rule_engine.evaluate_all(rules_df)

flagged_idx = rules_df.index[rule_result["fraud_r01"]][0]
case7_explanation = rule_engine.explain(rules_df.loc[flagged_idx])
print("Case 7 verdict:", case7_explanation["verdict_severity"], case7_explanation["verdict_rule_id"])

Case 7 verdict: CRITICAL fraud_r01


In [11]:
rag_result = pipeline.answer_for_flagged_transaction(db, case7_explanation)
print("OTOMATİK ÜRETİLEN SORU:", rag_result["question"])
print()
print("BULUNAN KAYNAKLAR:")
for r in rag_result["sources"]:
    print(f"  [{r.score:.3f}] {r.chunk.document_title}")
print()
print("NOT:", rag_result["note"])

OTOMATİK ÜRETİLEN SORU: A transaction was flagged by these rules: High Amount + Foreign Country + Night, New Device + New Address Together. Final verdict: CRITICAL / BLOCK. Explain why this transaction was considered risky, based on the relevant policies.

BULUNAN KAYNAKLAR:
  [0.107] Velocity (Hızlı Tekrar İşlem) Politikası
  [0.084] Yeni Kart, Yüksek Değerli İlk İşlem Politikası
  [0.084] Cihaz Parmak İzi Politikası

NOT: LLM generation failed (ollama:smollm2:360m): [Errno 111] Connection refused — showing retrieval + prompt only.


**Dürüst bir bulgu:** bu sorguda retrieval, coğrafi risk / mesai saatleri politikalarını değil,
velocity ve cihaz politikalarını üstte getirdi — beklenenden daha zayıf bir eşleşme. Sebep açık:
`answer_for_flagged_transaction`, Case 7'nin kural adlarını (`"High Amount + Foreign Country +
Night"`) kullanıyor ve bunlar **İngilizce** (projenin kod-İngilizce disipliniyle tutarlı), oysa
knowledge base dökümanları **Türkçe**. TF-IDF, tam kelime örtüşmesine dayandığı için bu dil
farkını köprüleyemiyor — "Foreign Country" ile "yabancı ülke" arasında hiçbir ortak kelime yok.

Bu, TF-IDF'in gerçek bir sınırı, gizlenmiyor: **Ollama'nın `all-minilm` gibi anlam-tabanlı
(semantic) bir embedding modeli, tam olarak bu tür durumlar için var** — kelime eşleşmesi değil
anlam yakınlığı ölçtüğü için diller arası ve eş-anlamlı ifadeler arası çok daha iyi genelleme
yapması beklenir. Bu, mimarideki `tfidf`→`ollama` geçişinin sadece mimari bir gösteriş değil,
somut bir kalite ihtiyacı olduğunun kanıtı.

## 8. Sağlayıcı Değişimi — DI Container ile `ollama`'ya geçiş (mimari doğrulama)

In [12]:
container.config.embedding_provider.from_value("ollama")
container.reset_singletons()  # Singleton provider'lar önbelleğe alınır — config değişikliğinin
                               # yeni bir örnek üretmesi için önbelleği temizlemek gerekiyor
ollama_pipeline = container.rag_pipeline()
print("yeni embedding provider:", ollama_pipeline.embedding_provider.name)

try:
    ollama_pipeline.embedding_provider.embed_query("test")
except Exception as exc:
    print(f"beklenen hata (Ollama henüz kurulu değil): {type(exc).__name__}: {exc}")


yeni embedding provider: ollama:all-minilm
beklenen hata (Ollama henüz kurulu değil): ConnectError: [Errno 111] Connection refused


In [13]:
close(db)
print("DB oturumu kapatıldı")

DB oturumu kapatıldı


Tek bir config değeri (`embedding_provider`) değiştirilerek `RAGPipeline` tamamen farklı bir
embedding stratejisine geçiyor — kod hiçbir yerde değişmedi. Ollama kurulunca bu hücre gerçek bir
embedding vektörü döndürecek; şu an sadece bağlantının reddedildiğini gösteriyor, mimarinin
çalıştığını değil beklemede olduğunu kanıtlıyor.

---

**Durum:** Case 8 (RAG Pipeline) mimarisi tam olarak kuruldu ve Ollama gerektirmeyen HER ŞEY
gerçekten doğrulandı: knowledge base (8 döküman, 6 gerçek + 2 deneysel), chunking, TF-IDF
embedding üretimi, SQLite'a persist (ve bu sırada bulunup düzeltilen gerçek bir cascade-delete
hatası), vector search (4/4 sorguda doğru döküman üstte), prompt/context injection (Builder
pattern), Case 7 köprüsü (otomatik soru üretimi), ve sağlayıcı-bağımsız zarif düşüş. Tek eksik:
gerçek LLM üretimi ve Ollama'nın semantic embedding'i — ikisi de Ollama kurulana kadar beklemede.
Kurulunca yapılacaklar: `ollama pull all-minilm`, `ollama pull smollm2:360m`, sonra bu notebook'un
7. ve 8. bölümleri gerçek sonuçlarla yeniden çalıştırılıp TF-IDF sonuçlarıyla karşılaştırılacak.